### Import Library & Load Dataset

In [70]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

df_raw = pd.read_csv('../dataset/jakarta_house.csv')
print(f"Dataset awal berhasil dimuat! Ukuran data: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")
display(df_raw.head())

Dataset awal berhasil dimuat! Ukuran data: 10000 baris, 9 kolom


,index,price,district,city,bed_rooms,bath_rooms,carport,land_area,building_area
0,1,5900000000,Citra Garden,Jakarta Barat,2,4,2,250.00,350.00
1,2,2700000000,Jelambar,Jakarta Barat,4,2,0,100.00,225.00
2,3,2200000000,Jelambar,Jakarta Barat,3,3,0,60.00,140.00
3,4,1900000000,Jelambar,Jakarta Barat,3,2,0,60.00,120.00
4,5,2100000000,Tanjung Duren,Jakarta Barat,4,3,0,56.00,108.00


### Memilih Fitur

In [71]:
kolom_digunakan = [
    'price', 
    'district', 
    'city', 
    'bed_rooms', 
    'bath_rooms', 
    'carport', 
    'land_area', 
    'building_area'
]

df_filtered = df_raw[kolom_digunakan].copy()

print(f"Hasil Column Filter ({df_filtered.shape[1]} kolom):")
display(df_filtered.head())

Hasil Column Filter (8 kolom):


,price,district,city,bed_rooms,bath_rooms,carport,land_area,building_area
0,5900000000,Citra Garden,Jakarta Barat,2,4,2,250.00,350.00
1,2700000000,Jelambar,Jakarta Barat,4,2,0,100.00,225.00
2,2200000000,Jelambar,Jakarta Barat,3,3,0,60.00,140.00
3,1900000000,Jelambar,Jakarta Barat,3,2,0,60.00,120.00
4,2100000000,Tanjung Duren,Jakarta Barat,4,3,0,56.00,108.00


### Penanganan Missing Value

In [72]:
print("Jumlah missing value sebelum ditangani:")
print(df_filtered.isnull().sum())

# Mengisi missing value pada land_area dan building_area menggunakan nilai median
df_filtered['land_area'] = df_filtered['land_area'].fillna(df_filtered['land_area'].median())
df_filtered['building_area'] = df_filtered['building_area'].fillna(df_filtered['building_area'].median())

print("\nJumlah missing value setelah ditangani:")
print(df_filtered.isnull().sum())
print(f"\nUkuran data setelah penanganan missing value: {df_filtered.shape[0]} baris, {df_filtered.shape[1]} kolom")


Jumlah missing value sebelum ditangani:
price            0
district         0
city             0
bed_rooms        0
bath_rooms       0
carport          0
land_area        3
building_area    5
dtype: int64

Jumlah missing value setelah ditangani:
price            0
district         0
city             0
bed_rooms        0
bath_rooms       0
carport          0
land_area        0
building_area    0
dtype: int64

Ukuran data setelah penanganan missing value: 10000 baris, 8 kolom


### Pembersihan Data Duplikat

In [73]:
print("Jumlah data duplikat:", df_filtered.duplicated().sum())

df_cleaned = df_filtered.drop_duplicates().reset_index(drop=True)

print(f"\nUkuran data setelah hapus duplikat: {df_cleaned.shape[0]} baris")
print("Jumlah data duplikat setelah penghapusan :", df_cleaned.duplicated().sum())

Jumlah data duplikat: 526

Ukuran data setelah hapus duplikat: 9474 baris
Jumlah data duplikat setelah penghapusan : 0


### Deteksi & Penanganan Outlier

In [74]:
kolom_numerik = [
    'price', 
    'bed_rooms', 
    'bath_rooms', 
    'carport', 
    'land_area', 
    'building_area'
]

# Menghitung Q1, Q3, dan IQR (1.5) pada kolom numerik
Q1 = df_cleaned[kolom_numerik].quantile(0.25)
Q3 = df_cleaned[kolom_numerik].quantile(0.75)
IQR = Q3 - Q1

batas_bawah = Q1 - 1.5 * IQR
batas_atas  = Q3 + 1.5 * IQR

# Mendeteksi outlier sebelum ditangani
kondisi_outlier = (df_cleaned[kolom_numerik] < batas_bawah) | (df_cleaned[kolom_numerik] > batas_atas)

tabel_sebelum = pd.DataFrame({
    'Jumlah Data': len(df_cleaned),
    'Jumlah Outlier': kondisi_outlier.sum(),
    'Batas Bawah': batas_bawah.round(2),
    'Batas Atas': batas_atas.round(2)
})
print("TABEL ANALISIS JUMLAH OUTLIER SEBELUM DITANGANI:")
display(tabel_sebelum)

# Menghapus baris yang memiliki nilai outlier
df_bersih_outlier = df_cleaned[~kondisi_outlier.any(axis=1)].reset_index(drop=True)

# Memeriksa kembali outlier pada batas yang sama setelah pembersihan
kondisi_outlier_setelah = (df_bersih_outlier[kolom_numerik] < batas_bawah) | (df_bersih_outlier[kolom_numerik] > batas_atas)

tabel_sesudah = pd.DataFrame({
    'Jumlah Data': len(df_bersih_outlier),
    'Jumlah Outlier': kondisi_outlier_setelah.sum(),
    'Batas Bawah': batas_bawah.round(2),
    'Batas Atas': batas_atas.round(2)
})
print("\nTABEL HASIL SETELAH PENANGANAN OUTLIER:")
display(tabel_sesudah)

print(f"\nTotal data sebelum penanganan outlier : {len(df_cleaned)} baris")
print(f"Total data setelah penanganan outlier : {len(df_bersih_outlier)} baris")

TABEL ANALISIS JUMLAH OUTLIER SEBELUM DITANGANI:


,Jumlah Data,Jumlah Outlier,Batas Bawah,Batas Atas
price,9474,1025,-8600000000.00,21000000000.00
bed_rooms,9474,291,0.00,8.00
bath_rooms,9474,256,-1.00,7.00
carport,9474,764,-0.50,3.50
land_area,9474,821,-255.00,697.00
building_area,9474,634,-250.00,790.00



TABEL HASIL SETELAH PENANGANAN OUTLIER:


,Jumlah Data,Jumlah Outlier,Batas Bawah,Batas Atas
price,7573,0,-8600000000.00,21000000000.00
bed_rooms,7573,0,0.00,8.00
bath_rooms,7573,0,-1.00,7.00
carport,7573,0,-0.50,3.50
land_area,7573,0,-255.00,697.00
building_area,7573,0,-250.00,790.00



Total data sebelum penanganan outlier : 9474 baris
Total data setelah penanganan outlier : 7573 baris


### Transformasi Logaritmik Target

In [75]:
X = df_bersih_outlier.drop(columns=['price'])
y = np.log1p(df_bersih_outlier['price'])

print("Perbandingan target sebelum dan sesudah transformasi logaritmik:")
display(pd.DataFrame({
    'Price Asli (Rp)': df_bersih_outlier['price'].head(),
    'Price Log': y.head()
}))

Perbandingan target sebelum dan sesudah transformasi logaritmik:


,Price Asli (Rp),Price Log
0,5900000000,22.50
1,2700000000,21.72
2,2200000000,21.51
3,1900000000,21.37
4,2100000000,21.47


### One-Hot Encoding Menggunakan Column Transformer

In [76]:
kolom_kategorikal = ['district', 'city']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), kolom_kategorikal)
    ],
    remainder='passthrough'
)

X_encoded_preview = preprocessor.fit_transform(X)

print(f"Jumlah fitur sebelum One-Hot Encoding: {X.shape[1]} fitur")
print(f"Jumlah fitur setelah One-Hot Encoding: {X_encoded_preview.shape[1]} fitur")

Jumlah fitur sebelum One-Hot Encoding: 7 fitur
Jumlah fitur setelah One-Hot Encoding: 251 fitur


### Partitioning Data (80% Training : 20% Testing)

In [77]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, shuffle=True
)

print(f"Jumlah Data Training (80%): {len(X_train)} baris")
print(f"Jumlah Data Testing  (20%): {len(X_test)} baris")

Jumlah Data Training (80%): 6058 baris
Jumlah Data Testing  (20%): 1515 baris


### Menyimpan Data Hasil Preprocessing

In [78]:
data_lengkap = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'preprocessor': preprocessor,
    'df_bersih': df_bersih_outlier
}

joblib.dump(data_lengkap, '../dataset/data_hasil_preprocessing.pkl')
print("✅ Data siap pemodelan berhasil disimpan ke '../dataset/data_hasil_preprocessing.pkl'!")

✅ Data siap pemodelan berhasil disimpan ke '../dataset/data_hasil_preprocessing.pkl'!
